# Tribunal de Justiça de São Paulo (TJSP)

[![Repo](https://img.shields.io/badge/GitHub-repo-blue?logo=github&logoColor=f5f5f5)](https://github.com/open-geodata/sp_tjsp_divadmin)

Necessário dar continuidade.


In [1]:
import concurrent.futures

import pandas as pd
from dotenv import load_dotenv

import open_geodata as geo

<br>

---

## Regiões Administrativas Judiciárias (RAJs)

Inicialmente, com auxílio do do pacote [lxml](https://lxml.de/), acessamos as informações da página [Quem Somos](https://www.tjsp.jus.br/QuemSomos/QuemSomos/RegioesAdministrativasJudiciarias) do TJSP.


In [2]:
tjsp = geo.sp.tjsp.div_admin.QuemSomos()
df_rajs = tjsp.get_raj()
df_rajs.info()
df_rajs.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id_raj              10 non-null     int64 
 1   raj_nome            10 non-null     object
 2   raj_sigla           10 non-null     object
 3   raj_regiao          10 non-null     object
 4   juiz_diretor_nome   10 non-null     object
 5   juiz_diretor_email  10 non-null     object
dtypes: int64(1), object(5)
memory usage: 612.0+ bytes


,id_raj,raj_nome,raj_sigla,raj_regiao,juiz_diretor_nome,juiz_diretor_email
0,1,1ª RAJ - Grande São Paulo,1ª RAJ,Grande São Paulo,Fernando Antonio Tasso,ftasso@tjsp.jus.br
1,2,2ª RAJ - Araçatuba,2ª RAJ,Araçatuba,Antonio Fernando Sanches Batagelo,abatagelo@tjsp.jus.br
2,3,3ª RAJ - Bauru,3ª RAJ,Bauru,Gilmar Ferraz Garmes,gilmargarmes@tjsp.jus.br
3,4,4ª RAJ - Campinas,4ª RAJ,Campinas,Renata Oliva Bernardes de Souza,rbsouza@tjsp.jus.br
4,5,5ª RAJ - Presidente Prudente,5ª RAJ,Presidente Prudente,Antonio Roberto Sylla,antoniosylla@tjsp.jus.br


<br>

Além das **_RAJs_**, obtemos também todas as comarcas, bem como qual a **_Circunscrição Judiciária_** que abrange aquela comarca.


In [3]:
df_comarcas = tjsp.get_comarcas()

# Ajusta Coluna
df_script1 = geo.sp.tjsp.div_admin.adjust_columns(
    df=df_comarcas, column_ajust='comarca_tjsp'
)

# Resultados
df_comarcas.info()
df_comarcas.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 321 entries, 0 to 320
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_cj              321 non-null    int64 
 1   comarca_tjsp       321 non-null    object
 2   comarca_tjsp_temp  321 non-null    object
dtypes: int64(1), object(2)
memory usage: 7.7+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 321 entries, 0 to 320
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_cj              321 non-null    int64 
 1   comarca_tjsp       321 non-null    object
 2   comarca_tjsp_temp  321 non-null    object
dtypes: int64(1), object(2)
memory usage: 7.7+ KB


,id_cj,comarca_tjsp,comarca_tjsp_temp
0,30,Adamantina,adamantina
1,50,Aguaí,aguai
2,32,Agudos,agudos
3,39,Altinópolis,altinopolis
4,53,Americana,americana


<br>

Já é possível relacionar as **_Circunscrição Judiciária_** às **_RAJs_**.


In [4]:
df_cj = tjsp.get_cj()
df_cj.info()
df_cj.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57 entries, 0 to 56
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id_cj     57 non-null     int64 
 1   cj_sigla  57 non-null     object
 2   cj_nome   57 non-null     object
 3   id_raj    57 non-null     int64 
dtypes: int64(2), object(2)
memory usage: 1.9+ KB


,id_cj,cj_sigla,cj_nome,id_raj
0,0,Capital,Capital,1
1,1,1ª CJ,1ª Circunscrição Judiciária,7
2,2,2ª CJ,2ª Circunscrição Judiciária,1
3,3,3ª CJ,3ª Circunscrição Judiciária,1
4,4,4ª CJ,4ª Circunscrição Judiciária,1


<br>

Agora só falta definir quais os municípios que pertencem a qual comarca.


<br>

---

## Lista Telefônica do TJSP

O TJSP tem tantas unicades espalhadas pelo Estado de São Paulo que há uma página dedicada à [Lista Telefônica](https://www.tjsp.jus.br/ListaTelefonica), onde digitando (a partir de) 3 caracteres, a página retorna 10 municípios sugeridos.


In [5]:
tjsp = geo.sp.tjsp.div_admin.TJSP()
tjsp.get_lista_municipios_tjsp(contain='San')

São 4508 termos para pesquisa


,id_municipio_tjsp,municipio_tjsp
0,6509,Águas de Santa Bárbara
1,6675,Espírito Santo do Pinhal
2,6676,Espírito Santo do Turvo
3,6892,Oscar Bressane
4,7000,Rosana
5,7014,Sandovalina
6,7015,Santa Adélia
7,7016,Santa Albertina
8,7017,Santa Bárbara d'Oeste
9,7018,Santa Branca


<br>

Como não sei como o TJSP escreve o nome dos municípios, optei por pegar o nome de todos os municípios do estado de São Paulo (com a grafia que eu ajustei) e iterar, a partir de 3 caracteres, a pesquisa.

Por exemplo, o município de "Piracicaba" será pesquisado a partir do 3º caractere.

- Pir
- Pira
- Pirac
- Piraci
- Piracic
- Piracica
- Piracicab
- Piracicaba

<br>

Para cada pesquisa será retornado os 10 municípios com a grafia mais semelhante e, por fim, obtenho os 645 municípios do Estado de São Paulo.


In [6]:
list_termos = tjsp.list_terms_search()

São 4508 termos para pesquisa


<br>

Para agilizar a pesquisa de mais de 4.000 termos (fragmentos de nomes de municípios), optei por usar o pacote [requests_ip_rotator](https://github.com/Ge0rg3/requests-ip-rotator), que instancia uma API na [AWS](https://aws.amazon.com/) para disparar desenas de requisições simultâneas.


In [7]:
load_dotenv()

AWS_ACCESS_KEY_ID = os.getenv('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = os.getenv('AWS_SECRET_ACCESS_KEY')

In [8]:
df = tjsp.search_terms_aws(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

df.info()
df.head()

Starting API gateway in 1 regions.
Using 1 endpoints with name 'https://www.tjsp.jus.br - IP Rotate API' (1 new).
Deleting gateway for site 'https://www.tjsp.jus.br'.
Deleted 1 endpoints with for site 'https://www.tjsp.jus.br'.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6813 entries, 0 to 6812
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_municipio_tjsp  6813 non-null   int64 
 1   municipio_tjsp     6813 non-null   object
dtypes: int64(1), object(1)
memory usage: 106.6+ KB


,id_municipio_tjsp,municipio_tjsp
0,6619,Cândido Rodrigues
1,6664,Duartina
2,7057,São José da Bela Vista
3,7052,São João da Boa Vista
4,7110,Teodoro Sampaio


<br>

Por fim obtenho a lista dos 645 municípios.


In [9]:
df_mun_tjsp = tjsp.adjust_data(df=df)
df_mun_tjsp.info()
df_mun_tjsp.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 645 entries, 0 to 644
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_municipio_tjsp  645 non-null    int64 
 1   municipio_tjsp     645 non-null    object
dtypes: int64(1), object(1)
memory usage: 10.2+ KB


,id_municipio_tjsp,municipio_tjsp
0,6504,Adamantina
1,6505,Adolfo
2,6506,Aguaí
3,6511,Agudos
4,6512,Alambari


<br>

Sei da existência de nome errados do TJSP e já corrijo.


In [10]:
df_mun_tjsp['municipio_tjsp'] = df_mun_tjsp['municipio_tjsp'].replace(
    {
        # Nome Errado (TJSP): Nome Correto (Michel)
        'Estrela dOeste': "Estrela d'Oeste",
        'Luís Antônio': 'Luiz Antônio',
        'Florínia': 'Florínea',
    }
)

<br>

Agora posso dar sequencia da unificação com os códigos do IBGE.

<br>

---

## Municípios TJSP

Inicialmente lemos a tabela que tem o nome correto dos municípios (eu que defini o que é correto ou não, acessando sites de prefeituras).


In [11]:
df_mun_sp = geo.data.load_dataset(db='sp', name='tab.municipio_nome')
df_mun_sp.head()

,id_municipio,municipio_nome
0,3500105,Adamantina
1,3500204,Adolfo
2,3500303,Aguaí
3,3500402,Águas da Prata
4,3500501,Águas de Lindóia


<br>

E concatenamos a tabela do TJSP com a tabela contendo o nome dos municípios.

In [12]:
df_mun = pd.merge(
    left=df_mun_sp,
    right=df_mun_tjsp,
    left_on='municipio_nome',
    right_on='municipio_tjsp',
    how='left',
)

# Confere se há algum erro
df_temp = df_mun[df_mun['municipio_tjsp'].isnull()]
if len(df_temp):
    raise Exception('Deu ruim')

# Results
df_mun.info()
df_mun.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 645 entries, 0 to 644
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_municipio       645 non-null    int64 
 1   municipio_nome     645 non-null    object
 2   id_municipio_tjsp  645 non-null    int64 
 3   municipio_tjsp     645 non-null    object
dtypes: int64(2), object(2)
memory usage: 20.3+ KB


,id_municipio,municipio_nome,id_municipio_tjsp,municipio_tjsp
0,3500105,Adamantina,6504,Adamantina
1,3500204,Adolfo,6505,Adolfo
2,3500303,Aguaí,6506,Aguaí
3,3500402,Águas da Prata,6507,Águas da Prata
4,3500501,Águas de Lindóia,6508,Águas de Lindóia


In [13]:
df_mun.to_csv(
    'tjsp_municipios.csv',
    index=False,
    encoding='utf-8',
)

In [14]:
df_mun = pd.read_csv('tjsp_municipios.csv')
df_mun

,id_municipio,municipio_nome,id_municipio_tjsp,municipio_tjsp
0,3500105,Adamantina,6504,Adamantina
1,3500204,Adolfo,6505,Adolfo
2,3500303,Aguaí,6506,Aguaí
3,3500402,Águas da Prata,6507,Águas da Prata
4,3500501,Águas de Lindóia,6508,Águas de Lindóia
...,...,...,...,...
640,3556909,Vista Alegre do Alto,7142,Vista Alegre do Alto
641,3556958,Vitória Brasil,7143,Vitória Brasil
642,3557006,Votorantim,7144,Votorantim
643,3557105,Votuporanga,7145,Votuporanga


<br>


-----

## Unidades

Ainda usando a Lista Telefônica do TJSP, encontrei uma função que é possível pesquisar pelo código (do TJSP) do município. Defini uma função que me retorna uma tabela contendo todas as unidades do TJSP (Fóruns, Setores, prédios etc) e, ainda, me informa a qual comarca o município pertence.

In [15]:
lista = geo.providers.sp.tjsp.div_admin.ListaTelefonica()
lista.get_lista_unidades_tjsp(cod_municipio=6504)

,id_municipio_tjsp,raj,municipio_tjsp,comarca_tjsp,comarca_sede,unidades
0,6504,5ª RAJ Presidente Prudente,Adamantina,Adamantina,1,Fórum I Adamantina
1,6504,5ª RAJ Presidente Prudente,Adamantina,Adamantina,1,Fórum II Adamantina (Residência Oficial)
2,6504,5ª RAJ Presidente Prudente,Adamantina,Adamantina,1,Fórum III Adamantina - UAAJ - FAC. ADAM. INTEG...


In [16]:
# Lista Municípios
list_cod_tjsp = list(df_mun['id_municipio_tjsp'])
list_cod_tjsp[:5]

[6504, 6505, 6506, 6507, 6508]

<br>

Iteramos as requisições dos 645 municípios.

In [17]:
MAX_THREADS = 5

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
    temp = executor.map(lista.get_lista_unidades_tjsp, list_cod_tjsp)
    df_com = pd.concat(list(temp), ignore_index=True)

# Results
df_com.info()
df_com.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1442 entries, 0 to 1441
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_municipio_tjsp  1442 non-null   int64 
 1   raj                1442 non-null   object
 2   municipio_tjsp     1442 non-null   object
 3   comarca_tjsp       1442 non-null   object
 4   comarca_sede       1442 non-null   int64 
 5   unidades           1442 non-null   object
dtypes: int64(2), object(4)
memory usage: 67.7+ KB


,id_municipio_tjsp,raj,municipio_tjsp,comarca_tjsp,comarca_sede,unidades
0,6504,5ª RAJ Presidente Prudente,Adamantina,Adamantina,1,Fórum I Adamantina
1,6504,5ª RAJ Presidente Prudente,Adamantina,Adamantina,1,Fórum II Adamantina (Residência Oficial)
2,6504,5ª RAJ Presidente Prudente,Adamantina,Adamantina,1,Fórum III Adamantina - UAAJ - FAC. ADAM. INTEG...
3,6505,8ª RAJ São José do Rio Preto,Adolfo,José Bonifácio,0,Fórum José Bonifácio - II (CEJUSC)
4,6505,8ª RAJ São José do Rio Preto,Adolfo,José Bonifácio,0,Fórum José Bonifácio I (Principal)


<br>

Montamos uma tabela contendo tudo.

In [18]:
# Merge
df_unidades = pd.merge(
    left=df_mun,
    right=df_com,
    left_on='id_municipio_tjsp',
    right_on='id_municipio_tjsp',
    how='inner',
    suffixes=['', '_copy'],
)

# Exclui coluna repetida
df_unidades = df_unidades.drop(
    labels=['municipio_tjsp_copy'],
    axis='columns',
    errors='ignore',
)

# Aplica strip em todo o dataframe
df_unidades = df_unidades.map(lambda x: x.strip() if isinstance(x, str) else x)

# Results
df_unidades.info()
df_unidades.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1442 entries, 0 to 1441
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_municipio       1442 non-null   int64 
 1   municipio_nome     1442 non-null   object
 2   id_municipio_tjsp  1442 non-null   int64 
 3   municipio_tjsp     1442 non-null   object
 4   raj                1442 non-null   object
 5   comarca_tjsp       1442 non-null   object
 6   comarca_sede       1442 non-null   int64 
 7   unidades           1442 non-null   object
dtypes: int64(3), object(5)
memory usage: 90.3+ KB


,id_municipio,municipio_nome,id_municipio_tjsp,municipio_tjsp,raj,comarca_tjsp,comarca_sede,unidades
0,3500105,Adamantina,6504,Adamantina,5ª RAJ Presidente Prudente,Adamantina,1,Fórum I Adamantina
1,3500105,Adamantina,6504,Adamantina,5ª RAJ Presidente Prudente,Adamantina,1,Fórum II Adamantina (Residência Oficial)
2,3500105,Adamantina,6504,Adamantina,5ª RAJ Presidente Prudente,Adamantina,1,Fórum III Adamantina - UAAJ - FAC. ADAM. INTEG...
3,3500204,Adolfo,6505,Adolfo,8ª RAJ São José do Rio Preto,José Bonifácio,0,Fórum José Bonifácio - II (CEJUSC)
4,3500204,Adolfo,6505,Adolfo,8ª RAJ São José do Rio Preto,José Bonifácio,0,Fórum José Bonifácio I (Principal)


<br>

-----

## Municípios

In [19]:
df_mun_com = df_unidades[
    [
        'id_municipio',
        'id_municipio_tjsp',
        'municipio_nome',
        'municipio_tjsp',
        #'raj',
        'comarca_tjsp',
        'comarca_sede',
        #'unidades',
    ]
]
df_mun_com = df_mun_com.drop_duplicates()
df_mun_com = df_mun_com.reset_index(drop=True)
df_mun_com.info()
df_mun_com.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 645 entries, 0 to 644
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_municipio       645 non-null    int64 
 1   id_municipio_tjsp  645 non-null    int64 
 2   municipio_nome     645 non-null    object
 3   municipio_tjsp     645 non-null    object
 4   comarca_tjsp       645 non-null    object
 5   comarca_sede       645 non-null    int64 
dtypes: int64(3), object(3)
memory usage: 30.4+ KB


,id_municipio,id_municipio_tjsp,municipio_nome,municipio_tjsp,comarca_tjsp,comarca_sede
0,3500105,6504,Adamantina,Adamantina,Adamantina,1
1,3500204,6505,Adolfo,Adolfo,José Bonifácio,0
2,3500303,6506,Aguaí,Aguaí,Aguaí,1
3,3500402,6507,Águas da Prata,Águas da Prata,São João da Boa Vista,0
4,3500501,6508,Águas de Lindóia,Águas de Lindóia,Águas de Lindóia,1


<br>

Crio uma tabela temporária, apenas para conseguir, posteriormente, trazer os códigos do IBGE para a tabela das comarcas


In [20]:
# Comarca
df_mun_com_sede = df_mun_com[df_mun_com['comarca_sede'] == 1]

# # Renomeia
# df_mun_com_sede = df_mun_com_sede.rename(
#     mapper={
#         'municipio_nome': 'comcarca_nome',
#         'id_municipio': 'id_comarca',
#     },
#     axis=1,
# )

# # Deleta Coluna
# df_mun_com_sede = df_mun_com_sede.drop(
#     labels=['comarca_sede', 'municipio_tjsp', 'comarca_tjsp'],
#     axis='columns',
#     errors='ignore',
# )

# Deleta Duplicados
df_mun_com_sede = df_mun_com_sede.drop_duplicates()

# Reset Index
df_mun_com_sede = df_mun_com_sede.reset_index(drop=True)

# Bata Bater
# df_tjsp_com = adjust_columns(df=df_tjsp_com, column_ajust='comarca_tjsp')

# Ajusta Coluna
df_mun_com_sede = geo.sp.tjsp.div_admin.adjust_columns(
    df=df_mun_com_sede, column_ajust='comarca_tjsp'
)

# Results
df_mun_com_sede.info()
df_mun_com_sede.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_municipio       320 non-null    int64 
 1   id_municipio_tjsp  320 non-null    int64 
 2   municipio_nome     320 non-null    object
 3   municipio_tjsp     320 non-null    object
 4   comarca_tjsp       320 non-null    object
 5   comarca_sede       320 non-null    int64 
 6   comarca_tjsp_temp  320 non-null    object
dtypes: int64(3), object(4)
memory usage: 17.6+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_municipio       320 non-null    int64 
 1   id_municipio_tjsp  320 non-null    int64 
 2   municipio_nome     320 non-null    object
 3   municipio_tjsp     320 non-null    object
 4   comarca_tjsp       320 non-null

,id_municipio,id_municipio_tjsp,municipio_nome,municipio_tjsp,comarca_tjsp,comarca_sede,comarca_tjsp_temp
0,3500105,6504,Adamantina,Adamantina,Adamantina,1,adamantina
1,3500303,6506,Aguaí,Aguaí,Aguaí,1,aguai
2,3500501,6508,Águas de Lindóia,Águas de Lindóia,Águas de Lindóia,1,aguas de lindoia
3,3500709,6511,Agudos,Agudos,Agudos,1,agudos
4,3501004,6515,Altinópolis,Altinópolis,Altinópolis,1,altinopolis


In [21]:
df_mun_com_sede['municipio_tjsp'].equals(df_mun_com_sede['comarca_tjsp'])
df_mun_com_sede[df_mun_com_sede['municipio_tjsp'] != df_mun_com_sede['comarca_tjsp']]

,id_municipio,id_municipio_tjsp,municipio_nome,municipio_tjsp,comarca_tjsp,comarca_sede,comarca_tjsp_temp
84,3515202,6677,Estrela d'Oeste,Estrela d'Oeste,Estrela dOeste,1,estrela doeste
